In [ ]:
import pandas as pd
from neo4j import GraphDatabase

# Configurações do Docker
URI = "bolt://localhost:7687"
USER = "neo4j"
PASSWORD = "itau1234"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

# Teste rápido para ver se ele acha o servidor
try:
    driver.verify_connectivity()
    print("Conectado ao Neo4j com sucesso!")
except Exception as e:
    print("Erro na conexão:", e)

✅ Conectado ao Neo4j com sucesso!


In [ ]:
# Lê o arquivo que você gerou no notebook anterior
df = pd.read_csv("extrato_simulado_itau.csv")

# A Query Cypher
query_cypher = """
MERGE (c:Cliente {id: $id_cliente})
MERGE (e:Estabelecimento {nome: $estabelecimento})
MERGE (cat:Categoria {nome: $categoria})

CREATE (t:Transacao {valor: $valor, data: $data, tipo: $tipo})

CREATE (c)-[:REALIZOU]->(t)
CREATE (t)-[:NO_ESTABELECIMENTO]->(e)
CREATE (t)-[:PERTENCE_A_CATEGORIA]->(cat)
"""

def limpar_e_injetar(tx, dataframe):
    # Limpa sujeiras anteriores
    tx.run("MATCH (n) DETACH DELETE n")

    # Loop injetando os dados
    for index, row in dataframe.iterrows():
        tx.run(query_cypher, 
               id_cliente=row['id_cliente'],
               estabelecimento=row['estabelecimento'],
               categoria=row['categoria'],
               valor=row['valor_reais'],
               data=row['data'],
               tipo=row['tipo'])

# Executa a função
with driver.session() as session:
    session.execute_write(limpar_e_injetar, df)
    print(f"{len(df)} transações injetadas no Neo4j!")

🚀 21 transações injetadas no Neo4j!
